In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import sys
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
from scipy.io import mmread
import pycats
from pandas.api.types import CategoricalDtype
import hotspot
import seaborn as sns
from scipy.stats import spearmanr
import statsmodels.api as sm

scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

In [ ]:
colors = ["#5DADE2", "white", "#FF69B4"]  # Lighter sky blue and hot pink
custom_cmap = LinearSegmentedColormap.from_list("light_skyblue_to_hot_pink", colors)

## Load data

In [ ]:
adata_full = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

fig_dir = "Reproducibility/Results/Plots/CD4_T/"
os.makedirs(fig_dir, exist_ok = True)

In [ ]:
tmp_subset = 'CD4_T'

adata = adata_full[adata_full.obs['lineage']==tmp_subset].copy()

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_latent_df.txt"
latent_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
latent_df = latent_df.loc[adata.obs_names, : ].values 
adata.obsm["MultiVI_latent"] = latent_df

tmp_path = f"Reproducibility/Data/embeddings/UC_DOGMA_{tmp_subset}_UMAP_df.txt"
UMAP_df = pd.read_csv(tmp_path,  sep='\t', index_col = 0)
adata.obsm["X_umap"] = UMAP_df.values

sc.pp.neighbors(adata, use_rep="MultiVI_latent")

In [ ]:
from pandas.api.types import CategoricalDtype

cat_type = CategoricalDtype(categories=["CD4_Tn","CD4_Tcm",'CD4_Tsen',"CD4_T_CD26","CD4_CTL","CD4_Th17","CD4_Tfh-like",
                                        "CD4_Tph-like","CD4_T_proliferative","Treg_naive","Treg_effector"], ordered=True)
adata.obs['celltype'] = adata.obs['celltype'].astype(cat_type)

## Visualization

### UMAP

In [ ]:
# Fig.5A
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['celltype'],
    frameon=False,
    #legend_loc="on data",
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}Figure5A_UMAP.pdf", bbox_inches='tight')
plt.close()

### RNA dotplot

In [ ]:
adata_RNA = adata[:, adata.var.modality == "Gene Expression"].copy()
adata_RNA.layers["counts"] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)

adata_RNA.layers['scaled'] = sc.pp.scale(adata_RNA, copy=True).X

In [ ]:
marker_genes = ["SELL","CCR7","TCF7",
                "IL7R",'NR4A1','HSPA1A','CD69',
                "TNF",'DPP4',"CCR6","IFNG","ITGAE","GZMA","GZMB",
                "IL17A",
                "PDCD1",'HAVCR2',"CXCL13","MKI67",
                "IKZF2","IL2RA","FOXP3","TNFRSF4","TNFRSF9","TNFRSF18","CCR8"
                ]

In [ ]:
# Fig.S7A
sc.pl.dotplot(adata_RNA, 
              marker_genes, 
              groupby = "celltype",
              dendrogram=False,
              cmap=custom_cmap,
              dot_max = 0.8,
              layer = 'scaled',
              use_raw=False,
              vmin = -0.5,
              vmax = 0.5,
              show = False)
plt.savefig(f"{fig_dir}FigureS9A_RNA_dotplot.pdf", bbox_inches='tight')
plt.close()

### Protein heatmap

In [ ]:
from muon import prot as pt
adt_df = adata_RNA.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata_RNA.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obsm["MultiVI_latent"] = adata.obsm["MultiVI_latent"]
pro_adata.obs["celltype"] = adata.obs["celltype"]

pt.pp.clr(pro_adata)

In [ ]:
marker_proteins = ['IL7Ra',
                   'KLRG1',
                   'CD26','TNFa',
                   'CD69','ITGB7','ITGAE','CD161',
                   'OX2','PD1','CD71',
                   'CD27',
                   'TIGIT','CD25','CTLA4',
                   'CCR8','41BB','CD177']

In [ ]:
# Fig.S7B
sc.pl.matrixplot(pro_adata, 
                 marker_proteins, 
                 groupby = "celltype",
                 dendrogram=False,
                 standard_scale='var',                 
                 cmap='plasma',
                 show = False)
plt.savefig(f"{fig_dir}FigureS9B_protein_heatmap.pdf", bbox_inches='tight')
plt.close()

### Signature score

In [ ]:
tmp_path = 'Reproducibility/Results/VISIONR/UC_DOGMA_CD4_T_signature_score_literature.txt'
sig_df = pd.read_csv(tmp_path, index_col=0, sep='\t')

adata.obs = pd.merge(left=adata.obs, right=sig_df.loc[adata.obs_names,:], how='left', left_index=True, right_index=True)

In [ ]:
# Fig.5B
sc.set_figure_params(figsize=(3, 3)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['Neoantigen-reactive'],
    color_map = 'magma',    
    vmax="p95",
    vmin='p5',
    use_raw=False,
    frameon=False,
    size = 3,
    wspace=0.1,
    legend_fontsize=1.5,
    show = False
)
plt.savefig(f"{fig_dir}Figure5B.pdf", bbox_inches='tight')
plt.close()

### Scatter plot between REL and TNFRSF genes

In [ ]:
rna_meta_ad = sc.read(f"Reproducibility/Results/SEACells/CD4_T/UC_DOGMA_SEACells_rna_meta_ad.h5ad")
sc.pp.normalize_total(rna_meta_ad)
sc.pp.log1p(rna_meta_ad)
sc.pp.scale(rna_meta_ad)

In [ ]:
eRegulon_df = pd.read_csv("Reproducibility/Results/scenicplus/CD4_T/outs/eRegulon_df.txt", index_col=0, sep='\t')
eRegulon_df = eRegulon_df.loc[rna_meta_ad.obs_names,:]
rna_meta_ad.obs = pd.merge(left=rna_meta_ad.obs, right=eRegulon_df, how='left', left_index=True, right_index=True)

In [ ]:
# Function to plot correlation
def plot_corr(x, y, xlabel, ylabel):
    rho, pval = spearmanr(x, y)
    sns.regplot(x=x, y=y, scatter_kws={'s':10, 'alpha':0.5}, line_kws={"color":"red"}, lowess=True)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"Spearman ρ={rho:.2f}, p={pval:.2e}")

In [ ]:
# Extract the REL_extended scores and TNFRSF genes
df = pd.DataFrame(index=rna_meta_ad.obs_names)
df["REL_extended"] = rna_meta_ad.obs["REL_extended_+/+_(288g)"]

# TNFRSF family gene list (you can extend)
tnfrsf_genes = ["TNFRSF4",'TNFRSF8', "TNFRSF9", "TNFRSF18"]
for g in tnfrsf_genes:
    if g in rna_meta_ad.var_names:
        df[g] = rna_meta_ad[:, g].X.toarray().flatten()

# Loop over TNFRSF genes
for g in tnfrsf_genes:
    if g in df.columns:
        plt.figure(figsize=(4,4))
        plot_corr(df["REL_extended"], df[g], "REL_extended", g)
        plt.show()

In [ ]:
comet = ["#E6E7E8","#3A97FF","#8816A7","black"]
comet_cmap = LinearSegmentedColormap.from_list('custom',comet)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

tnfrsf_genes = list(dict.fromkeys(["TNFRSF9","TNFRSF4","TNFRSF18","TNFRSF8"]))

with PdfPages(f"{fig_dir}Figure5E_and_S9J.pdf") as pdf:
    for gene in tnfrsf_genes:
        x = df["REL_extended"]; y = df[gene]
        mask = x.notna() & y.notna()
        x = x[mask]; y = y[mask]
        rho, p = scipy.stats.spearmanr(x, y)
        g = sns.jointplot(x=x, y=y,kind="kde", space=0, fill=True, thresh=0,
                          levels=100, cmap=comet_cmap, marginal_kws={'color':'#3A97FF'})
        # fill background with first color of cmap
        g.ax_joint.set_facecolor(comet_cmap(0))
        g.ax_joint.add_patch(plt.Rectangle((x.min(), y.min()), x.max()-x.min(), y.max()-y.min(),color=comet_cmap(0), zorder=-1))
        g.ax_joint.text(0.04, 0.96, fr"rho = {rho:.2f}, p = {p:.2e}",
                        transform=g.ax_joint.transAxes, ha="left", va="top",
                        fontsize=11, bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.8))
        g.ax_joint.set_xlabel("REL eRegulon score")
        g.ax_joint.set_ylabel(f'Scaled {gene} expression')
        pdf.savefig(g.fig)
        plt.close(g.fig)

### Pseudotime UMAP

In [ ]:
Treg_meta = pd.read_csv("Reproducibility/Results/CellRank2/CD4_T/Treg_dpt_pseudotime.txt", sep='\t', index_col=0)

adata.obs['dpt_pseudotime'] = adata.obs.index.map(Treg_meta['dpt_pseudotime'])

In [ ]:
sc.set_figure_params(figsize=(4, 4)) 
sc.set_figure_params(vector_friendly=True, dpi_save=100)

cmap = plt.get_cmap("gnuplot").copy()
cmap.set_bad(color="#F2F2F2")

# Plot with modified colormap
sc.pl.embedding(
    adata,
    basis="umap",
    color=["dpt_pseudotime"],
    color_map=cmap,
    legend_loc="on data",
    show=False
)

plt.savefig(f"{fig_dir}Figure5D_UMAP_pseudotime.pdf", bbox_inches='tight')
plt.close()